#02_silver_beneficiarios.sql

Camada Silver - dim_beneficiarios

Este notebook transforma bronze.cadastro_beneficiario (raw/string) em uma dimensão silver limpa e valida padronizando demografia e atributos cadastrais para análises (churn, segmentação, experiência).

##Objetivos
- tipa data_nascimento (DATE) e idade (INT)
- normaliza sexo (M/F/N/I) e UF (sigla válida)
- deduplica por beneficiario_id (mais recente ingestion_ts)
- rejeita registros sem chave e com data_nascimento inválida (futura/parse inválido)
- persiste idempotente via MERGE usando row_hash
- roda incremental via checkpoint (silver_ops.pipeline_checkpoint)

##Estratégia
- SQL-first com TEMP VIEWs
- TRY_CAST para conversões resilientes
- dicionário simples via CASE para normalização de sexo
- row_number() para dedupe determinístico
- rejects com motibo
- MERGE com row_hash

##Chave natural

beneficiario_id

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Pipeline checkpoint

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING
)
USING DELTA;

##Tabelas de destino (silver + rejects)

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.dim_beneficiarios (
  beneficiario_id STRING,
  cpf_hash STRING,
  data_nascimento DATE,
  idade INT,
  sexo STRING,
  uf STRING,
  municipio STRING,
  canal_aquisicao STRING,
  segmento_vinculo STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.dim_beneficiarios (
  beneficiario_id STRING,
  cpf_hash STRING,
  data_nascimento STRING,
  idade STRING,
  sexo STRING,
  uf STRING,
  municipio STRING,
  canal_aquisicao STRING,
  segmento_vinculo STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Leitura incremental do bronze (via checkpoint)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_benef_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'dim_beneficiarios'
)
SELECT *
FROM healthcare_dev.bronze.cadastro_beneficiarios
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem + padronização

In [0]:
CREATE OR REPLACE TEMP VIEW stg_benef_typed AS
SELECT
  cast(beneficiario_id as string) AS beneficiario_id,
  cast(cpf_hash as string) AS cpf_hash,

  -- date: se não parsear vira NULL
  try_cast(data_nascimento as date) AS data_nascimento,

  -- idade: se inválida vira NULL
  try_cast(idade as int) AS idade,

  -- normalização de sexo
  CASE
    WHEN sexo IS NULL OR trim(sexo) = '' THEN 'N/I'
    WHEN upper(trim(sexo)) IN ('M','MASC','MASCULINO') THEN 'M'
    WHEN upper(trim(sexo)) IN ('F','FEM','FEMININO') THEN 'F'
    ELSE 'N/I'
  END AS sexo,

  upper(trim(cast(uf as string))) AS uf,
  initcap(trim(cast(municipio as string))) AS municipio,
  upper(trim(cast(canal_aquisicao as string))) AS canal_aquisicao,
  upper(trim(cast(segmento_vinculo as string))) AS segmento_vinculo,

  ingestion_ts,
  load_id,
  source_file
FROM stg_benef_raw;

##Deduplicação determinística (beneficiario_id)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_benef_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY beneficiario_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_benef_typed
) x
WHERE rn = 1;

##Regras de qualidade (classificação + motivo)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_benef_classified AS
SELECT
  *,
  CASE
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN data_nascimento IS NOT NULL AND data_nascimento > current_date() THEN 'BIRTHDATE_IN_FUTURE'
    WHEN uf IS NOT NULL AND uf <> '' AND uf NOT IN (
      'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
      'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
    ) THEN 'INVALID_UF'
    ELSE NULL
  END AS reject_reason
FROM stg_benef_dedup;

##Persiste rejects

In [0]:
INSERT INTO healthcare_dev.silver_rejects.dim_beneficiarios
SELECT
  beneficiario_id,
  cpf_hash,
  cast(data_nascimento as string) AS data_nascimento,
  cast(idade as string) AS idade,
  sexo,
  uf,
  municipio,
  canal_aquisicao,
  segmento_vinculo,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_benef_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash (MERGE idempotente)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_benef_valid AS
SELECT
  beneficiario_id,
  cpf_hash,
  data_nascimento,
  idade,
  sexo,
  uf,
  municipio,
  canal_aquisicao,
  segmento_vinculo,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    beneficiario_id,
    coalesce(cpf_hash,''),
    coalesce(cast(data_nascimento as string),''),
    coalesce(cast(idade as string),''),
    coalesce(sexo,''),
    coalesce(uf,''),
    coalesce(municipio,''),
    coalesce(canal_aquisicao,''),
    coalesce(segmento_vinculo,'')
  ), 256) AS row_hash
FROM stg_benef_classified
WHERE reject_reason IS NULL;

##MERGE na silver

In [0]:
MERGE INTO healthcare_dev.silver.dim_beneficiarios AS tgt
USING stg_benef_valid AS src
ON tgt.beneficiario_id = src.beneficiario_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'dim_beneficiarios' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_benef_valid;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_silver_beneficiarios
FROM healthcare_dev.silver.dim_beneficiarios;

In [0]:
SELECT COUNT(*) AS n_rejects_beneficiarios
FROM healthcare_dev.silver_rejects.dim_beneficiarios;

In [0]:
SELECT reject_reason, COUNT(*) AS qtd
FROM healthcare_dev.silver_rejects.dim_beneficiarios
GROUP BY reject_reason
ORDER BY qtd DESC;